# 📊 Análise de Componentes Principais (PCA) para Elaboração de Rankings Socioeconômicos

## 📌 Contexto do Projeto
O objetivo deste projeto é elaborar um ranking dos distritos do município de São Paulo, ordenando-os com base em indicadores socioeconômicos coletados pela prefeitura. A base de dados contém múltiplas variáveis numéricas e, para evitar uma análise isolada ou arbitrária baseada em apenas um único indicador, é utilizada a técnica de **PCA (Principal Component Analysis)** para consolidar todas as informações simultaneamente em um índice estruturado.

---

## 🧠 Fundamentos Teóricos do PCA

### 1. Definição da Técnica
O PCA é uma técnica de **aprendizado não-supervisionado**. Isso significa que os dados não possuem uma variável alvo (*target*) predefinida. Em vez de realizar previsões de valores futuros, o método executa uma **transformação matemática** nos dados originais, convertendo as variáveis em **fatores independentes (componentes principais)**.

* **Ordenação de Variância:** O primeiro fator gerado explica a maior parcela da variabilidade dos dados. O segundo fator retém a segunda maior parcela, e assim sucessivamente.
* **Limite de Componentes:** A quantidade máxima de fatores gerados é exatamente igual ao número de variáveis numéricas originais presentes no conjunto de dados.

### 2. Principais Aplicações do PCA
* **Redução de Dimensionalidade:** Substitui um grande volume de variáveis originais por apenas alguns fatores principais, reduzindo o tamanho do dataset sem perda significativa de informação.
* **Remoção de Multicolinearidade:** Elimina problemas onde as variáveis originais são excessivamente correlacionadas entre si. Os novos fatores gerados são totalmente independentes (ortogonais) e não possuem correlação mútua.
* **Elaboração de Rankings:** Permite consolidar múltiplos indicadores complexos em um índice único de pontuação para ordenação de observações.

### ⚠️ Restrições Técnicas
O algoritmo do PCA fundamenta-se estritamente na construção de uma **Matriz de Correlação**. Por consequência, a técnica **aceita apenas variáveis numéricas**. Variáveis categóricas ou textuais (como identificadores e nomes de regiões) devem ser isoladas do cálculo estatístico principal.


In [7]:
import pandas as pd


In [ ]:
distritos = pd.read_csv("Dados/distritos_sp.csv")


In [9]:
distritos


,cod_ibge,distritos,renda,quota,escolaridade,idade,mortalidade,txcresc,causasext,favel,denspop
0,1,Água Rasa,1961,34.619999,7.6,32,13.86,-1.840000,52.980000,0.00,125.610001
1,12,Alto de Pinheiros,4180,75.959999,8.4,33,8.68,-2.520000,38.570000,0.69,57.560001
2,23,Anhanguera,1093,4.500000,5.8,23,15.36,18.120001,22.680000,0.00,8.570000
3,34,Aricanduva,1311,21.020000,6.8,27,18.43,-1.070000,76.220001,5.38,138.539993
4,45,Artur Alvim,1248,15.910000,7.0,27,19.73,-1.400000,67.250000,4.11,167.399994
...,...,...,...,...,...,...,...,...,...,...,...
91,92,Vila Medeiros,1405,19.760000,6.8,27,15.43,-1.410000,77.980003,2.49,188.929993
92,93,Vila Prudente,1755,32.080002,7.2,30,14.36,-2.550000,66.510002,7.43,101.440002
93,94,Vila Sônia,2970,41.410000,7.4,27,16.76,-0.900000,74.680000,14.93,80.120003
94,95,São Domingos,2047,23.510000,6.8,26,14.30,0.710000,62.349998,8.55,72.919998


In [10]:
distritos.info()

<class 'pandas.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   cod_ibge      96 non-null     int64  
 1   distritos     96 non-null     str    
 2   renda         96 non-null     int64  
 3   quota         96 non-null     float64
 4   escolaridade  96 non-null     float64
 5   idade         96 non-null     int64  
 6   mortalidade   96 non-null     float64
 7   txcresc       96 non-null     float64
 8   causasext     96 non-null     float64
 9   favel         96 non-null     float64
 10  denspop       96 non-null     float64
dtypes: float64(7), int64(3), str(1)
memory usage: 8.4 KB


In [12]:
variaveis_numericas = distritos.drop(['cod_ibge', 'distritos'], axis=1)
variaveis_numericas.corr()

,renda,quota,escolaridade,idade,mortalidade,txcresc,causasext,favel,denspop
renda,1.000000,0.920099,0.777332,0.732307,-0.519585,-0.424711,-0.462516,-0.146957,-0.019711
quota,0.920099,1.000000,0.850455,0.832737,-0.520282,-0.554767,-0.491020,-0.243010,0.057374
escolaridade,0.777332,0.850455,1.000000,0.955825,-0.582601,-0.692968,-0.606621,-0.432548,0.157673
idade,0.732307,0.832737,0.955825,1.000000,-0.553758,-0.703237,-0.615073,-0.499838,0.141469
mortalidade,-0.519585,-0.520282,-0.582601,-0.553758,1.000000,0.346049,0.422790,0.130877,-0.093018
txcresc,-0.424711,-0.554767,-0.692968,-0.703237,0.346049,1.000000,0.234472,0.281853,-0.279084
causasext,-0.462516,-0.491020,-0.606621,-0.615073,0.422790,0.234472,1.000000,0.404447,-0.045281
favel,-0.146957,-0.243010,-0.432548,-0.499838,0.130877,0.281853,0.404447,1.000000,-0.106481
denspop,-0.019711,0.057374,0.157673,0.141469,-0.093018,-0.279084,-0.045281,-0.106481,1.000000


# 📉 Calculando a Estatística KMO (Kaiser-Meyer-Olkin)

## 📌 O que é o Critério KMO?
O teste de **Kaiser-Meyer-Olkin (KMO)** é um índice estatístico que avalia se o conjunto de dados é adequado para a aplicação de uma análise fatorial ou PCA. Ele mede a consistência geral dos dados, indicando a proporção da variância das variáveis que pode ser considerada comum (atribuída a fatores compartilhados).

---

## 📊 Interpretação dos Resultados
A estatística KMO varia estritamente em um intervalo de **0 a 1**:

* **Próximos de 1:** Indicam que as variáveis compartilham um percentual de variância bastante elevado (correlações de Pearson altas). Significa que os dados estão **adequadamente ajustados** para o modelo.
* **Próximos de 0:** Decorrem de correlações de Pearson baixas entre as variáveis. Isso serve como um sinal de alerta, indicando que a análise fatorial ou PCA será **inadequada** para esse conjunto de dados.


In [14]:
from factor_analyzer.factor_analyzer import calculate_kmo


In [15]:
kmo_variaveis, kmo = calculate_kmo(variaveis_numericas)

In [16]:
kmo_variaveis

array([0.77821831, 0.81941916, 0.8560973 , 0.81750459, 0.94677797,
       0.84146713, 0.89083164, 0.78871213, 0.63275248])

In [17]:
kmo

np.float64(0.833091424182929)

# 📉 Teste de Esfericidade de Bartlett

## 📌 Objetivo do Teste
O **Teste de Esfericidade de Bartlett** avalia a adequabilidade dos dados para a análise fatorial ou PCA. Ele faz isso comparando a **matriz de correlações ($\rho$)** das variáveis originais com uma **matriz identidade ($I$)** de mesma dimensão.

* **Matriz Identidade ($I$):** É uma matriz que possui o número 1 na diagonal principal e zero em todas as outras posições. Caso nossos dados fossem iguais a ela, significaria que a correlação entre as variáveis é nula.

---

## 🔬 Hipóteses Estatísticas
O teste avalia duas hipóteses principais:

* **Hipótese Nula ($H_0$):** $\rho = I$
  A matriz de correlações é igual à matriz identidade. Indica que as variáveis **não possuem correlação significativa** entre si. Se não há correlação, a extração de fatores do PCA **não será adequada**.
* **Hipótese Alternativa ($H_1$):** $\rho \neq I$
  A matriz de correlações é significativamente diferente da matriz identidade. Indica que as variáveis **possuem correlações significativas** entre si, validando a aplicação do PCA.

---

## 🎯 Regra de Decisão (O que buscamos?)
Para podermos seguir com o PCA, o nosso objetivo é **rejeitar a Hipótese Nula ($H_0$)**. 

Na prática do código, isso é avaliado através do **p-valor (p-value)**. Buscamos um **p-valor menor que 0.05** (considerando um nível de significância clássico de 5%), o que comprova que as correlações fora da diagonal principal são estatisticamente diferentes de zero.

---

## ⚠️ Observação Metodológica Crucial
Deve-se **sempre preferir o Teste de Esfericidade de Bartlett à estatística KMO** para fins de decisão definitiva sobre a adequação global da análise fatorial ou PCA. 

* **Bartlett:** É um teste de hipóteses formal, fundamentado em uma distribuição de probabilidades determinada e níveis rígidos de significância estatística.
* **KMO:** Funciona apenas como um coeficiente (estatística descritiva), calculado sem uma distribuição de probabilidades associada, o que impossibilita a avaliação formal de um nível de significância correspondente para a tomada de decisão.


In [19]:
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity


In [20]:
qui_quadrado, p_valor = calculate_bartlett_sphericity(variaveis_numericas)

In [21]:
print(qui_quadrado)
print(p_valor)

748.1593126421544
5.607017481839493e-134


In [23]:
p_valor < 0.05


np.True_